# Session 1: Basic Concepts of a Network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-01-basic-network-concepts.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-01_encapsulation.py`

In [1]:
"""Session 1 - Encapsulation through the TCP/IP and OSI layers (simulation).

Shows how application data gains a header at each layer on the way DOWN the
sending stack, and loses it again on the way UP the receiving stack.
Nothing is sent on a real network: it is a teaching model only.
Run:  python session-01_encapsulation.py
"""

# Each layer: (OSI number, OSI name, PDU name, header added by that layer)
LAYERS = [
    (7, "Application",  "Data",    "HTTP: GET /index.html"),
    (6, "Presentation", "Data",    "TLS: encrypted record"),
    (5, "Session",      "Data",    "Session id=42"),
    (4, "Transport",    "Segment", "TCP: src=50123 dst=443 seq=1"),
    (3, "Network",      "Packet",  "IP: 10.0.0.5 -> 93.184.216.34 TTL=64"),
    (2, "Data Link",    "Frame",   "ETH: aa:aa:aa:00:00:01 -> bb:bb:bb:00:00:fe"),
    (1, "Physical",     "Bits",    "signal on the wire"),
]


def encapsulate(payload: str) -> list[str]:
    """Wrap the payload in one header per layer, top (7) to bottom (1)."""
    stack = [payload]
    unit = payload
    for num, name, pdu, header in LAYERS:
        if num == 1:                       # physical layer turns the frame into bits
            size = len(unit)
            unit = f"010110... (the {size}-character frame sent as signals)"
        else:
            unit = f"[{header}] {unit}"
            size = len(unit)
        stack.append(unit)
        print(f"L{num} {name:<12} PDU={pdu:<8} size={size:>3}  {unit[:60]}")
    return stack


def decapsulate(stack: list[str]) -> str:
    """Receiver reads from the bottom up, removing one header per layer."""
    print("\nReceiver strips headers bottom-up:")
    for (num, name, pdu, header), unit in zip(reversed(LAYERS), reversed(stack[1:])):
        print(f"L{num} {name:<12} reads its header -> passes {pdu.lower()} up")
    return stack[0]


if __name__ == "__main__":
    data = "Hello, lab server!"
    print("Sender encapsulates top-down:")
    s = encapsulate(data)
    delivered = decapsulate(s)
    overhead = len(s[-2]) - len(data)
    print(f"\nDelivered to the application: {delivered!r}")
    print(f"Header overhead before the wire: {overhead} characters "
          f"for {len(data)} characters of real data ({overhead / len(data):.0%})")

Sender encapsulates top-down:
L7 Application  PDU=Data     size= 42  [HTTP: GET /index.html] Hello, lab server!
L6 Presentation PDU=Data     size= 66  [TLS: encrypted record] [HTTP: GET /index.html] Hello, lab s
L5 Session      PDU=Data     size= 82  [Session id=42] [TLS: encrypted record] [HTTP: GET /index.ht
L4 Transport    PDU=Segment  size=113  [TCP: src=50123 dst=443 seq=1] [Session id=42] [TLS: encrypt
L3 Network      PDU=Packet   size=152  [IP: 10.0.0.5 -> 93.184.216.34 TTL=64] [TCP: src=50123 dst=4
L2 Data Link    PDU=Frame    size=198  [ETH: aa:aa:aa:00:00:01 -> bb:bb:bb:00:00:fe] [IP: 10.0.0.5 
L1 Physical     PDU=Bits     size=198  010110... (the 198-character frame sent as signals)

Receiver strips headers bottom-up:
L1 Physical     reads its header -> passes bits up
L2 Data Link    reads its header -> passes frame up
L3 Network      reads its header -> passes packet up
L4 Transport    reads its header -> passes segment up
L5 Session      reads its header -> passes data up


## `session-01_topology.py`

In [2]:
"""Session 1 - Compare network topologies by cable count and fault tolerance.

For each topology we build the list of links between N devices, then ask:
  * how many cables / links are needed?
  * if any ONE device fails, do the remaining devices stay connected?
That second question finds the single points of failure (SPOF).
Run:  python session-01_topology.py
"""
from itertools import combinations


def build(topology: str, n: int) -> set[tuple[int, int]]:
    """Return the set of links (a, b) for n devices numbered 0..n-1."""
    if topology == "star":            # device 0 is the central switch
        return {(0, i) for i in range(1, n)}
    if topology == "ring":
        return {tuple(sorted((i, (i + 1) % n))) for i in range(n)}
    if topology == "full mesh":
        return set(combinations(range(n), 2))
    if topology == "bus":             # model the shared cable as node 0 (backbone)
        return {(0, i) for i in range(1, n)}
    if topology == "tree":            # binary tree: parent of i is (i-1)//2
        return {((i - 1) // 2, i) for i in range(1, n)}
    raise ValueError(topology)


def connected(nodes: set[int], links: set[tuple[int, int]]) -> bool:
    """Breadth-first search: can every node reach every other node?"""
    if not nodes:
        return True
    start = next(iter(nodes))
    seen, queue = {start}, [start]
    while queue:
        cur = queue.pop(0)
        for a, b in links:
            for x, y in ((a, b), (b, a)):
                if x == cur and y in nodes and y not in seen:
                    seen.add(y)
                    queue.append(y)
    return seen == nodes


def single_points_of_failure(n: int, links) -> list[int]:
    """Nodes whose failure splits the rest of the network."""
    spof = []
    for victim in range(n):
        rest = set(range(n)) - {victim}
        rest_links = {l for l in links if victim not in l}
        if not connected(rest, rest_links):
            spof.append(victim)
    return spof


if __name__ == "__main__":
    N = 6
    print(f"{'Topology':<10} {'Links':>5}  {'Formula':<12} Single points of failure")
    formulas = {"star": "n-1", "ring": "n", "full mesh": "n(n-1)/2",
                "bus": "1 backbone", "tree": "n-1"}
    for topo in ["bus", "star", "ring", "tree", "full mesh"]:
        links = build(topo, N)
        spof = single_points_of_failure(N, links)
        label = "none" if not spof else ", ".join(f"node {s}" for s in spof)
        print(f"{topo:<10} {len(links):>5}  {formulas[topo]:<12} {label}")

    print("\nFull-mesh cost grows fast:")
    for n in (4, 10, 50, 100):
        print(f"  {n:>3} sites -> {n * (n - 1) // 2:>5} links")

Topology   Links  Formula      Single points of failure
bus            5  1 backbone   node 0
star           5  n-1          node 0
ring           6  n            none
tree           5  n-1          node 0, node 1, node 2
full mesh     15  n(n-1)/2     none

Full-mesh cost grows fast:
    4 sites ->     6 links
   10 sites ->    45 links
   50 sites ->  1225 links
  100 sites ->  4950 links
